# Pothole Detection -- YOLO26n Training, Validation & Testing

**Phase 1 scope only** -- this notebook trains and evaluates a single-class
"pothole" detector. It does NOT build any backend/frontend/API.

**Dataset source:** [Kaggle -- Annotated Potholes Image Dataset](https://www.kaggle.com/dsv/973710)
(this versioned download link resolves to `chitholian/annotated-potholes-dataset`;
we use the stable dataset slug with the official Kaggle API rather than the
numeric `dsv` id, which is not a valid Kaggle CLI target)

**Known public metadata (verify against the actual downloaded files in Step 3 -- do not trust this blindly):**
- 665 images with matching Pascal VOC XML annotations (NOT already YOLO format)
- A `splits.json` file defines an 80% train / 20% test split -- **no validation split is provided**
- Single class in the XML `<name>` tags (exact string verified programmatically below, not assumed)
- `<path>` tags inside the XML may be stale; use `<filename>` instead (this notebook does)

**Because there is no official validation split**, we carve one out of the
provided 80% training portion (85/15) ourselves, so we end up with
train/val/test while keeping the original 20% test set fully held out and
untouched as the final "unseen data" evaluation -- this is the alternative
evaluation procedure required for Phase 1's Step 6.

**Run this in Google Colab with a GPU runtime**: `Runtime > Change runtime type > T4 GPU`.

**Credential setup (do this before running, do NOT paste your key into any cell):**
1. Get a Kaggle API token: Kaggle -> your profile -> Settings -> API -> "Create New Token" (downloads `kaggle.json`)
2. Open `kaggle.json` locally and copy its `username` and `key` fields
3. In this Colab notebook, open the key icon in the left sidebar ("Secrets")
4. Add two secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY`, toggle "Notebook access" on for both
5. Do not commit `kaggle.json` or these values anywhere in the repository


## Step 1 -- Install dependencies

In [ ]:
!pip install -q "ultralytics>=8.3" kaggle
import ultralytics
ultralytics.checks()


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, then re-run this cell.")


## Step 2 -- Download the dataset via the Kaggle API (authenticated, not scraped)

In [ ]:
import os, json
from google.colab import userdata

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_username = userdata.get("KAGGLE_USERNAME")
kaggle_key = userdata.get("KAGGLE_KEY")
assert kaggle_username and kaggle_key, "Set KAGGLE_USERNAME and KAGGLE_KEY Colab secrets first (see markdown above)."

with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

!kaggle datasets download -d chitholian/annotated-potholes-dataset -p /content/raw_pothole --unzip


## Step 3 -- Inspect the raw dataset

In [ ]:
import glob, json

RAW_DIR = "/content/raw_pothole"
print("Top-level contents:", os.listdir(RAW_DIR))

images = sorted(glob.glob(os.path.join(RAW_DIR, "**", "*.jpg"), recursive=True))
xmls = sorted(glob.glob(os.path.join(RAW_DIR, "**", "*.xml"), recursive=True))
print(f"Images found: {len(images)}")
print(f"XML annotation files found: {len(xmls)}")

splits_path = glob.glob(os.path.join(RAW_DIR, "**", "splits.json"), recursive=True)
if splits_path:
    with open(splits_path[0]) as f:
        splits = json.load(f)
    print("splits.json keys:", list(splits.keys()))
    for k, v in splits.items():
        if isinstance(v, list):
            print(f"  {k}: {len(v)} entries")
else:
    print("No splits.json found -- will fall back to a random 80/20 split.")
    splits = None


## Step 4 -- Verify labels and class names (mandatory -- do not skip)

Parse every XML annotation and collect every distinct `<name>` value that
appears. We do NOT assume it already says exactly `pothole`.


In [ ]:
import xml.etree.ElementTree as ET
from collections import Counter

def parse_voc(xml_path):
    root = ET.parse(xml_path).getroot()
    filename = root.findtext("filename")
    size = root.find("size")
    width = int(size.findtext("width"))
    height = int(size.findtext("height"))
    objs = []
    for obj in root.findall("object"):
        name = obj.findtext("name").strip()
        bnd = obj.find("bndbox")
        xmin = float(bnd.findtext("xmin")); ymin = float(bnd.findtext("ymin"))
        xmax = float(bnd.findtext("xmax")); ymax = float(bnd.findtext("ymax"))
        objs.append((name, xmin, ymin, xmax, ymax))
    return filename, width, height, objs

class_name_counts = Counter()
parsed = {}
for xp in xmls:
    filename, w, h, objs = parse_voc(xp)
    parsed[xp] = (filename, w, h, objs)
    for (name, *_ ) in objs:
        class_name_counts[name] += 1

print("Distinct class name strings found across all XML files:")
for name, cnt in class_name_counts.most_common():
    print(f"  {name!r}: {cnt} instances")


## Step 5 -- Class normalization (explicit, no silent discarding)

Canonical target: exactly one class, `pothole` (index 0). Any class-name
string found above that is a clear synonym (case/pluralization variants of
"pothole") is mapped to the canonical name. If a genuinely different,
unrelated class name appears, the cell stops and requires a manual decision
rather than silently dropping or merging it.


In [ ]:
CANONICAL_NAME = "pothole"
SYNONYMS = {"pothole", "potholes", "Pothole", "Potholes", "pot-hole", "pot_hole"}

found_names = set(class_name_counts.keys())
unexpected = {n for n in found_names if n not in SYNONYMS}

if unexpected:
    print("UNEXPECTED / NON-SYNONYM CLASS NAMES FOUND -- manual review required:")
    for n in sorted(unexpected):
        print(f"  {n!r}: {class_name_counts[n]} instances")
    raise SystemExit(
        "Resolve unexpected class names above before proceeding: decide explicitly whether "
        "to include them as additional hazard classes or intentionally exclude them, and document why."
    )
else:
    print(f"All discovered class names are synonyms of '{CANONICAL_NAME}'. Proceeding with a single-class dataset.")
    print("Found:", found_names)


## Step 6 -- Convert Pascal VOC XML -> YOLO txt format

In [ ]:
import shutil, random

YOLO_DIR = "/content/pothole_yolo"
for split in ("train", "val", "test"):
    os.makedirs(os.path.join(YOLO_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DIR, split, "labels"), exist_ok=True)

def voc_to_yolo_line(xmin, ymin, xmax, ymax, w, h, cls_id=0):
    cx = ((xmin + xmax) / 2) / w
    cy = ((ymin + ymax) / 2) / h
    bw = (xmax - xmin) / w
    bh = (ymax - ymin) / h
    return f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"

# Build filename -> image path lookup (splits.json references filenames, not full paths).
image_lookup = {os.path.basename(p): p for p in images}

def xml_name_to_jpg(fname):
    # splits.json lists ANNOTATION (.xml) filenames per Kaggle's own README,
    # e.g. "img-565.xml" -- but image_lookup keys are .jpg filenames, so we
    # must convert before comparing or every membership test silently fails
    # and every image falls through to the "train" bucket.
    return os.path.splitext(fname)[0] + ".jpg"

if splits and "train" in splits and "test" in splits:
    test_files = {xml_name_to_jpg(f) for f in splits["test"]}
    trainval_files = {xml_name_to_jpg(f) for f in splits["train"]} - test_files
else:
    all_files = list(image_lookup.keys())
    random.seed(42)
    random.shuffle(all_files)
    n_test = int(0.2 * len(all_files))
    test_files = set(all_files[:n_test])
    trainval_files = set(all_files[n_test:])

trainval_files = sorted(trainval_files)
random.seed(42)
random.shuffle(trainval_files)
n_val = int(0.15 * len(trainval_files))
val_files = set(trainval_files[:n_val])
train_files = set(trainval_files[n_val:])

print(f"train={len(train_files)} val={len(val_files)} test={len(test_files)}")
assert len(test_files) > 0, "Test split is empty -- splits.json parsing likely broke (check filename/extension matching above)."

def assign_split(fname):
    if fname in test_files:
        return "test"
    if fname in val_files:
        return "val"
    return "train"

converted, skipped = 0, 0
for xp, (filename, w, h, objs) in parsed.items():
    img_key = filename if filename in image_lookup else os.path.basename(xp).replace(".xml", ".jpg")
    if img_key not in image_lookup:
        skipped += 1
        continue
    split = assign_split(img_key)
    img_src = image_lookup[img_key]
    img_dst = os.path.join(YOLO_DIR, split, "images", img_key)
    shutil.copy(img_src, img_dst)

    lines = [voc_to_yolo_line(xmin, ymin, xmax, ymax, w, h) for (_, xmin, ymin, xmax, ymax) in objs]
    label_name = os.path.splitext(img_key)[0] + ".txt"
    with open(os.path.join(YOLO_DIR, split, "labels", label_name), "w") as f:
        f.write("\n".join(lines))
    converted += 1

print(f"Converted {converted} images, skipped {skipped} (no matching image file found).")


## Step 7 -- Write the YOLO data config

In [ ]:
DATA_YAML = os.path.join(YOLO_DIR, "data.yaml")
data_cfg = {
    "path": YOLO_DIR,
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "nc": 1,
    "names": ["pothole"],
}
import yaml
with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data_cfg, f)
print(open(DATA_YAML).read())


## Step 8 -- Load the model

In [ ]:
from ultralytics import YOLO

# YOLO26n (nano) is the current lightweight Ultralytics model (released Jan 2026,
# ~2.4M params, NMS-free end-to-end head) -- chosen for fast prototype training and
# fast CPU/edge inference later. Falls back to YOLO11n if the installed
# ultralytics version does not yet ship YOLO26 weights.
MODEL_WEIGHTS = "yolo26n.pt"
try:
    model = YOLO(MODEL_WEIGHTS)
except Exception as e:
    print(f"Could not load {MODEL_WEIGHTS} ({e}); falling back to yolo11n.pt")
    MODEL_WEIGHTS = "yolo11n.pt"
    model = YOLO(MODEL_WEIGHTS)
print("Loaded:", MODEL_WEIGHTS)


## Step 9 -- Train

In [ ]:
EPOCHS = 100
IMGSZ = 640
BATCH = 16
PATIENCE = 20

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    project="runs/pothole",
    name="train",
    exist_ok=True,
    plots=True,
)


## Step 10 -- Validate (on the VAL split we carved out, separate from training)

In [ ]:
val_metrics = model.val(data=DATA_YAML, split="val")

print("=== Validation metrics ===")
print("precision(B):", val_metrics.box.mp)
print("recall(B):   ", val_metrics.box.mr)
print("mAP50:       ", val_metrics.box.map50)
print("mAP50-95:    ", val_metrics.box.map)

# Ultralytics writes confusion_matrix.png, PR_curve.png, etc. into the val run dir.
import glob
val_dir = val_metrics.save_dir
print("Validation artifacts saved to:", val_dir)
for p in sorted(glob.glob(str(val_dir) + "/*.png")):
    print(" -", p)


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

cm_path = str(val_metrics.save_dir / "confusion_matrix.png")
try:
    img = Image.open(cm_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion matrix (validation set)")
    plt.show()
except FileNotFoundError:
    print("confusion_matrix.png not found -- with a single class it may not be generated by all Ultralytics versions.")


## Step 11 -- Test on unseen images (original Kaggle 20% TEST split, never used for train or val)

In [ ]:
# Final held-out TEST evaluation -- these images were never used for training or
# validation. This is the mandatory "unseen data" evaluation for Phase 1.
test_metrics = model.val(data=DATA_YAML, split="test")

print("=== TEST metrics (unseen images) ===")
print("precision(B):", test_metrics.box.mp)
print("recall(B):   ", test_metrics.box.mr)
print("mAP50:       ", test_metrics.box.map50)
print("mAP50-95:    ", test_metrics.box.map)
print("Artifacts:   ", test_metrics.save_dir)


In [ ]:
import glob, random
from ultralytics import YOLO

test_images = sorted(glob.glob("os.path.join(YOLO_DIR, 'test', 'images', '*.*')"))
print(f"Found {len(test_images)} unseen test images")

sample = random.sample(test_images, min(8, len(test_images)))
pred_results = model.predict(source=sample, conf=0.25, save=True, project="runs/predict", name="samples", exist_ok=True)

import matplotlib.pyplot as plt
from PIL import Image

n = len(pred_results)
cols = min(4, n) if n else 1
rows = (n + cols - 1) // cols if n else 1
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = axes.flatten() if n > 1 else [axes]
for ax, res in zip(axes, pred_results):
    im = Image.fromarray(res.plot()[..., ::-1])
    ax.imshow(im)
    ax.axis("off")
    n_det = len(res.boxes)
    ax.set_title(f"{n_det} detection(s)")
for ax in axes[len(pred_results):]:
    ax.axis("off")
plt.suptitle("Sample predictions on unseen TEST images -- hazard: pothole")
plt.tight_layout()
plt.show()


## Step 12 -- Metrics summary (real numbers only -- do not hand-edit)

In [ ]:
import json

summary = {
    "hazard": "pothole",
    "model_weights": MODEL_WEIGHTS,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "val_precision": float(val_metrics.box.mp),
    "val_recall": float(val_metrics.box.mr),
    "val_map50": float(val_metrics.box.map50),
    "val_map50_95": float(val_metrics.box.map),
    "test_precision": float(test_metrics.box.mp),
    "test_recall": float(test_metrics.box.mr),
    "test_map50": float(test_metrics.box.map50),
    "test_map50_95": float(test_metrics.box.map),
}
print(json.dumps(summary, indent=2))

with open("training_summary_pothole.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved training_summary_pothole.json -- copy this into docs/PHASE1.md's results table as real evidence.")


## Step 13 -- Export best.pt

In [ ]:
import shutil, os

HAZARD = "pothole"
best_pt_src = str(model.trainer.best) if hasattr(model, "trainer") and model.trainer is not None else "runs/pothole/train/weights/best.pt"
print("Best weights produced at:", best_pt_src)

# Intended final repo-relative location (NOT committed to git -- see docs/PHASE1.md):
#   ai/models/pothole/best.pt
local_target_dir = f"ai_models_{HAZARD}"
os.makedirs(local_target_dir, exist_ok=True)
local_target = os.path.join(local_target_dir, "best.pt")
shutil.copy(best_pt_src, local_target)
print("Copied to:", local_target)

# Optional: persist to Google Drive so the weight survives the Colab session.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_dir = f"/content/drive/MyDrive/ResQDrive/models/{HAZARD}"
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy(best_pt_src, os.path.join(drive_dir, "best.pt"))
    print("Also copied to Google Drive:", drive_dir)
except Exception as e:
    print("Drive not mounted / not in Colab -- skipping Drive copy:", e)

# Download to your machine (place it at ai/models/pothole/best.pt in the repo).
try:
    from google.colab import files
    files.download(local_target)
except Exception as e:
    print("files.download unavailable in this environment:", e)


## Done

Copy `training_summary_pothole.json`'s numbers into `docs/PHASE1.md`'s
results table, and place the downloaded `best.pt` at
`ai/models/pothole/best.pt` in the repo (git-ignored by design).
